In [5]:
!pip install datasets transformers[torch] pandas pillow scikit-learn

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Define your local paths
train_path = r"C:\Users\NeuraGroww\Downloads\Hackathon\train.csv"
test_path = r"C:\Users\NeuraGroww\Downloads\Hackathon\test.csv"

# 2. Load the CSVs
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 3. Clean and Prepare
# We keep only 'text' and 'target' for training
train_df = train_df[['text', 'target']].copy()

# Remove URLs from the text to reduce noise
train_df['text'] = train_df['text'].str.replace(r'http\S+', '', regex=True)
test_df['text'] = test_df['text'].str.replace(r'http\S+', '', regex=True)

# 4. Split into Training and Validation sets (90/10 split)
# This replaces the 'train_test_split' from the datasets library
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.1, 
    random_state=42, 
    stratify=train_df['target']
)

print(f"✅ Loaded {len(train_data)} training rows and {len(val_data)} validation rows.")
print(train_data.head())

✅ Loaded 6851 training rows and 762 validation rows.
                                                   text  target
6867  Trauma can happen anywhere -- school home etc....       1
5188  @breakingnewslh @bree_mars watch cnn's the sev...       1
5413  ?You should be scared. You should be screaming...       0
5499  when you are quarantined to a little corner bc...       0
2308  @AngusMacNeilSNP Every case for Yes has been u...       0


In [5]:
import torch
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset

# 1. Load the pre-trained DistilBERT Tokenizer
# This downloads the vocabulary map that DistilBERT was originally trained on
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 2. Create a fast PyTorch Dataset Class
class DisasterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize the text: pad short sentences, truncate long ones
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 3. Convert your Pandas dataframes into these PyTorch datasets
train_dataset = DisasterDataset(
    texts=train_data['text'].tolist(),
    labels=train_data['target'].tolist(),
    tokenizer=tokenizer
)

val_dataset = DisasterDataset(
    texts=val_data['text'].tolist(),
    labels=val_data['target'].tolist(),
    tokenizer=tokenizer
)

print("✅ Tokenization complete! Data is ready for the model.")
# Let's look at the first converted tweet
print(train_dataset[0])

C:\Users\NeuraGroww\anaconda3\envs\drug_final\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tokenization complete! Data is ready for the model.
{'input_ids': tensor([  101, 12603,  2064,  4148,  5973,  1011,  1011,  2082,  2188,  4385,
         1012,  1011,  1011,  2012,  2151,  2051,  1012,  4553,  1996,  5925,
         1005,  1055,  1997, 12603,  1998,  2129,  2000,  6687,  1012,  1012,
         1012,  1057,  1035,   102,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,    

In [11]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Load the pre-trained model with a classification head
# num_labels=2 because we have two targets: 0 (No Disaster) and 1 (Disaster)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2 
)

# 2. Define a metric function so we can see the accuracy while it trains
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predictions)}

# 3. Configure training for your 6GB GPU
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=,              # Keep it fast for the hackathon
    per_device_train_batch_size=16,  # 16 is a safe size for 6GB VRAM
    per_device_eval_batch_size=32,
    warmup_steps=100,                
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",           # Evaluate at the end of each epoch
    save_strategy="epoch",
    fp16=True,                       # MAGIC BULLET: mixed precision for speed/low memory
    report_to="none"                 # Disables third-party logging prompts
)

# 4. Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,     # Using the dataset you prepped earlier
    eval_dataset=val_dataset,        # Using the validation set you prepped earlier
    compute_metrics=compute_metrics
)

# 5. FIRE IT UP! Start training
print("🚀 Starting training... Please wait for the progress bar to finish.")
trainer.train()

# 6. THIS IS THE CRITICAL STEP: Save the model to your hard drive!
trainer.save_model('./disaster_model_final')
tokenizer.save_pretrained('./disaster_model_final')
print("✅ Model successfully saved to ./disaster_model_final")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting training... Please wait for the progress bar to finish.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.403800,0.409305,0.814961
2,0.301200,0.431316,0.832021


✅ Model successfully saved to ./disaster_model_final


In [12]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# 1. Dynamically get the current working directory of this notebook
current_dir = os.getcwd()
model_path = os.path.join(current_dir, "disaster_model_final")

print(f"Looking for model folder at: {model_path}")

# 2. The Ultimate Sanity Check
if not os.path.exists(model_path):
    print("❌ ERROR: The folder DOES NOT EXIST here!")
    print("Here is what is actually in your current folder:")
    print(os.listdir(current_dir))
    print("\n👉 ACTION REQUIRED: Go back and re-run the training cell. Make sure it finishes and prints '✅ Model saved to ./disaster_model_final'")
else:
    print("✅ Folder found! Bypassing HF bug and loading model...")
    
    # 3. Load the objects safely
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

    classifier = pipeline(
        "text-classification", 
        model=model, 
        tokenizer=tokenizer
    )

    # 4. Test it
    test_tweet = "Massive flooding on 19th Main Road near the Jaynagar BDA complex! Water is waist-deep, we need a rescue boat now! #BangaloreRains"
    emergency_result = classifier(test_tweet)

    print("\n🚨 EMERGENCY TWEET TEST:")
    print(f"Output: {emergency_result}")

Looking for model folder at: C:\Users\NeuraGroww\Downloads\Hackathon\disaster_model_final
✅ Folder found! Bypassing HF bug and loading model...


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



🚨 EMERGENCY TWEET TEST:
Output: [{'label': 'LABEL_1', 'score': 0.9918198585510254}]


In [13]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# 1. Dynamically get the exact path to the folder you just created
current_dir = os.getcwd()
model_path = os.path.join(current_dir, "disaster_model_final")
print(f"Loading model from: {model_path}")

# 2. EXPLICITLY load the objects (local_files_only=True guarantees it won't check the web)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

# 3. Initialize the pipeline with your locally loaded objects
classifier = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer
)

# 4. Write custom, localized test tweets for your demo
emergency_tweet = "Massive flooding on 19th Main Road near the Jaynagar BDA complex! Water is waist-deep, we need a rescue boat now! #BangaloreRains"
casual_tweet = "Just had the best dosas near the Jaynagar BDA complex. The weather is so nice today!"

# 5. Run the predictions!
print("\n🚨 EMERGENCY TWEET TEST:")
print(f"Input: {emergency_tweet}")
print(f"Output: {classifier(emergency_tweet)}")

print("\n☕ NORMAL TWEET TEST:")
print(f"Input: {casual_tweet}")
print(f"Output: {classifier(casual_tweet)}")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Loading model from: C:\Users\NeuraGroww\Downloads\Hackathon\disaster_model_final

🚨 EMERGENCY TWEET TEST:
Input: Massive flooding on 19th Main Road near the Jaynagar BDA complex! Water is waist-deep, we need a rescue boat now! #BangaloreRains
Output: [{'label': 'LABEL_1', 'score': 0.9918198585510254}]

☕ NORMAL TWEET TEST:
Input: Just had the best dosas near the Jaynagar BDA complex. The weather is so nice today!
Output: [{'label': 'LABEL_1', 'score': 0.5772352814674377}]


In [14]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

current_dir = os.getcwd()
model_path = os.path.join(current_dir, "disaster_model_final")

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

# Added top_k=None to force it to return the exact probability for BOTH labels
classifier = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer,
    top_k=None 
)

def check_tweet(text, threshold=0.80):
    # Get all scores
    results = classifier(text)[0]
    
    # Find the specific score for LABEL_1 (Disaster)
    disaster_score = 0.0
    for r in results:
        if r['label'] == 'LABEL_1':
            disaster_score = r['score']
            
    # Apply our custom 80% hackathon threshold
    if disaster_score >= threshold:
        return f"🚨 EMERGENCY (Confidence: {disaster_score*100:.1f}%)"
    else:
        return f"☕ NORMAL (Disaster Probability: {disaster_score*100:.1f}%) - Ignored by system."

# Test it again!
emergency_tweet = "Massive flooding on 19th Main Road near the Jaynagar BDA complex! Water is waist-deep, we need a rescue boat now! #BangaloreRains"
casual_tweet = "Just had the best dosas near the Jaynagar BDA complex. The weather is so nice today!"

print("--- JAYNAGAR SYSTEM TEST ---")
print(f"Input 1: {emergency_tweet}")
print(check_tweet(emergency_tweet))

print(f"\nInput 2: {casual_tweet}")
print(check_tweet(casual_tweet))

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


--- JAYNAGAR SYSTEM TEST ---
Input 1: Massive flooding on 19th Main Road near the Jaynagar BDA complex! Water is waist-deep, we need a rescue boat now! #BangaloreRains
🚨 EMERGENCY (Confidence: 99.2%)

Input 2: Just had the best dosas near the Jaynagar BDA complex. The weather is so nice today!
☕ NORMAL (Disaster Probability: 57.7%) - Ignored by system.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# 1. PyTorch Preprocessing (Replaces AutoImageProcessor)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Standard ResNet normalization values
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
])

# 2. Load Data (Replaces HF load_dataset)
data_dir = r"C:\Users\NeuraGroww\Downloads\Hackathon\image_data"

# Assuming your image_data folder has 'train' and 'test' subfolders
train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), data_transforms)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), data_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 3. Load Native PyTorch ResNet18 (Replaces AutoModelForImageClassification)
# This downloads the reliable PyTorch weights
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 4. Swap the final classification layer for your specific number of classes
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(num_ftrs, num_classes)

# Move model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Native PyTorch ResNet-18 ready for {num_classes} classes on {device}!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\NeuraGroww/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:13<00:00, 3.51MB/s]


Native PyTorch ResNet-18 ready for 2 classes on cuda:0!


In [3]:
import torch.optim as optim
import time

# 1. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
# Adam optimizer with a small learning rate is great for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 2. The Training Loop
num_epochs = 3 # Kept at 3 for quick hackathon iterations

start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    # Each epoch has a training and validation phase
    for phase in ['train', 'test']:
        if phase == 'train':
            model.train()  # Set model to training mode
            dataloader = train_loader
        else:
            model.eval()   # Set model to evaluate mode
            dataloader = test_loader

        running_loss = 0.0
        running_corrects = 0

        # Iterate over data
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            # Track history only if in train phase
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Backward pass + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Calculate epoch loss and accuracy
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    print()

time_elapsed = time.time() - start_time
print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

# 3. Save the Model Weights for your Demo
torch.save(model.state_dict(), 'native_disaster_vision_model.pth')
print("Model weights saved to 'native_disaster_vision_model.pth'")

Epoch 1/3
---------------
Train Loss: 0.0863 | Acc: 0.9675
Test Loss: 0.0916 | Acc: 0.9630

Epoch 2/3
---------------
Train Loss: 0.0114 | Acc: 0.9969
Test Loss: 0.0189 | Acc: 0.9918

Epoch 3/3
---------------
Train Loss: 0.0178 | Acc: 0.9942
Test Loss: 0.1524 | Acc: 0.9692

Training complete in 1m 28s
Model weights saved to 'native_disaster_vision_model.pth'


In [4]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import torch.nn.functional as F
import os

# 1. Setup Device and Classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# EXACT MATCH: PyTorch sorted your folders alphabetically
class_names = ['Damaged', 'Normal'] 

# 2. Re-initialize the Model Architecture
print("Loading model architecture...")
model = models.resnet18(weights=None) 
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names)) 

# 3. Load Your Custom Saved Weights
model.load_state_dict(torch.load('native_disaster_vision_model.pth', map_location=device))
model = model.to(device)
model.eval() 
print("Weights loaded successfully!")

# 4. Image Preprocessing (Identical to training)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 5. The Inference Function
def test_single_image(image_path):
    print(f"\n--- Testing Image: {os.path.basename(image_path)} ---")
    
    # Load and preprocess the image
    image = Image.open(image_path).convert('RGB')
    input_tensor = preprocess(image)
    input_batch = input_tensor.unsqueeze(0).to(device) 

    # Run the model
    with torch.no_grad():
        output = model(input_batch)
        
    # Convert raw output logits to probabilities
    probabilities = F.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probabilities, 0)
    
    predicted_class = class_names[predicted_idx.item()]
    
    # Custom print logic based on your exact class names
    if predicted_class == 'Damaged': 
        print(f"🚨 ALERT: {predicted_class.upper()} INFRASTRUCTURE DETECTED")
    else:
        print(f"✅ STATUS: {predicted_class.upper()} (No action needed)")
        
    print(f"Confidence: {confidence.item() * 100:.2f}%\n")
    
    return predicted_class, confidence.item()

# ==========================================
# Run Your Test Here!
# ==========================================
# Automatically grab an image from your Damaged test folder to verify it works
damaged_test_folder = r"C:\Users\NeuraGroww\Downloads\Hackathon\image_data\test\Damaged"

if os.path.exists(damaged_test_folder):
    # Get all files in the folder
    test_files = os.listdir(damaged_test_folder)
    if test_files:
        # Construct the full path to the first image in that folder
        test_image_path = os.path.join(damaged_test_folder, test_files[0])
        test_single_image(test_image_path)
    else:
        print(f"\nYour folder {damaged_test_folder} is empty. Put an image in it to test!")
else:
    print(f"\nCould not find the folder: {damaged_test_folder}")

Loading model architecture...
Weights loaded successfully!

--- Testing Image: 559.jpg ---
🚨 ALERT: DAMAGED INFRASTRUCTURE DETECTED
Confidence: 91.72%



C:\Users\NeuraGroww\AppData\Local\Temp\ipykernel_2896\817345604.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('native_disaster_vision